In [1]:
# Importing Libraries
import pandas as pd
import numpy as np
import joblib
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported")

Libraries imported


In [2]:
# Loading New Dataset
print("Loading final_bn_data.csv...")

try:
    new_df = pd.read_csv('../data/final_bn_data.csv')
    print("Successfully loaded with default encoding")
except UnicodeDecodeError:
    encodings = ['utf-8', 'latin1', 'cp1252']
    for encoding in encodings:
        try:
            new_df = pd.read_csv('../data/final_bn_data.csv', encoding=encoding)
            print(f"Successfully loaded with {encoding} encoding")
            break
        except:
            continue

print(f"Dataset shape: {new_df.shape}")
print(f"Columns: {new_df.columns.tolist()}")
print(f"First few rows:")
print(new_df.head())

Loading final_bn_data.csv...
Successfully loaded with default encoding
Dataset shape: (14537, 4)
Columns: ['category', 'headline', 'content', 'label']
First few rows:
   category                                        headline  \
0  National                              ৮ দিনে ১৮ বিল পাস!   
1    Sports    আ’লীগের জনসভায় লোকে লোকারণ্য ফেনী ট্রাংক রোড   
2  National  মাদ্রাসায় জোড়া খুন: পরিচালক তিন দিনের রিমান্ডে   
3    Sports        নেপালকে হারিয়ে গ্রুপ চ্যাম্পিয়ন বাংলাদেশ   
4  National             কুড়িগ্রামে ২ শিক্ষার্থীর লাশ উদ্ধার   

                                             content  label  
0  দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত ...    0.0  
1  একাদশ জাতীয় সংসদ নির্বাচনকে সামনে রেখে সাংগঠনি...    0.0  
2  গাজীপুরে জোড়া খুন মামলার প্রধান আসামি মাদ্রাসা...    0.0  
3  সাফ অনূর্ধ্ব-১৮ নারী ফুটবল চ্যাম্পিয়নশিপে নেপা...    1.0  
4  কুড়িগ্রাম প্রতিনিধি : কুড়িগ্রাম সদর উপজেলার বে...    1.0  


In [3]:
# Analyzing Dataset Structure
print("Analyzing dataset structure...")

if 'text' not in new_df.columns:
    print("'text' column not found!")
    print("Available columns:", new_df.columns.tolist())

    text_candidates = []
    for col in new_df.columns:
        if len(new_df) > 0:
            sample = str(new_df[col].iloc[0])
            if len(sample) > 50:
                text_candidates.append((col, len(sample)))
                print(f" '{col}' might be text (length: {len(sample)})")

    if text_candidates:
        text_column = max(text_candidates, key=lambda x: x[1])[0]
        print(f"Using '{text_column}' as text column")
        new_df = new_df.rename(columns={text_column: 'text'})
else:
    print("'text' column found")

if 'label' in new_df.columns:
    print("'label' column found")
    print(f"Label distribution:\n{new_df['label'].value_counts()}")
else:
    print("No 'label' column - will only make predictions")

if 'text' in new_df.columns and len(new_df) > 0:
    sample_text = new_df['text'].iloc[0]
    print(f"\n Text sample (first 200 chars):")
    print(sample_text[:200])
    print(f"Text length: {len(sample_text)} characters")

Analyzing dataset structure...
'text' column not found!
Available columns: ['category', 'headline', 'content', 'label']
 'content' might be text (length: 2853)
Using 'content' as text column
'label' column found
Label distribution:
label
1.0    10000
0.0     4537
Name: count, dtype: int64

 Text sample (first 200 chars):
দশম জাতীয় সংসদের মেয়াদ শেষ হয়ে যাচ্ছে। কার্যত আর মাত্র একটি অধিবেশনই রয়েছে বর্তমান সরকারের। অক্টোবরের মাঝামাঝিতে সংক্ষিপ্ত একটি অধিবেশন ডেকেই ‍ইিত টানা হবে এ সংসেদর। এরপর ডিসেম্বরের শেষ সপ্তাহে একাদশ 
Text length: 2853 characters


In [4]:
# Loading Models and Vectorizer
print("Loading trained models and vectorizer...")

with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
print("Vectorizer loaded")

models = {
    'Naive_Bayes': joblib.load('naive_bayes.pkl'),
    'Random_Forest': joblib.load('random_forest.pkl'),
    'LightGBM': joblib.load('lightgbm.pkl'),
    'Complement_NB': joblib.load('complement_nb.pkl'),
    'Passive_Aggressive': joblib.load('passive_aggressive.pkl')
}
print("All models loaded")

Loading trained models and vectorizer...
Vectorizer loaded
All models loaded


In [5]:
# Making Predictions
print("Making predictions...")

if 'text' not in new_df.columns:
    print("No text column available for predictions")
else:
    X_new = vectorizer.transform(new_df['text'])
    print(f"Transformed shape: {X_new.shape}")

    all_predictions = {}
    for model_name, model in models.items():
        predictions = model.predict(X_new)
        all_predictions[model_name] = predictions
        new_df[f'pred_{model_name}'] = predictions
        print(f"{model_name} predictions completed")

    print("\nPrediction Summary:")
    for model_name in models.keys():
        pred_counts = new_df[f'pred_{model_name}'].value_counts().to_dict()
        print(f"{model_name:<20}: {pred_counts}")

Making predictions...
Transformed shape: (14537, 5000)
Naive_Bayes predictions completed
Random_Forest predictions completed
LightGBM predictions completed
Complement_NB predictions completed
Passive_Aggressive predictions completed

Prediction Summary:
Naive_Bayes         : {1: 13234, 0: 1303}
Random_Forest       : {1: 13310, 0: 1227}
LightGBM            : {1: 13254, 0: 1283}
Complement_NB       : {1: 11536, 0: 3001}
Passive_Aggressive  : {1: 12868, 0: 1669}


C:\Users\imtia\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# Calculating Accuracy
print("Evaluating results...")

if 'label' in new_df.columns:
    accuracy_results = {}
    print("\n Accuracy on New Data:")
    for model_name in models.keys():
        accuracy = accuracy_score(new_df['label'], new_df[f'pred_{model_name}'])
        accuracy_results[model_name] = accuracy
        print(f"{model_name:<20}: {accuracy:.4f} ({accuracy*100:.2f}%)")

    best_model = max(accuracy_results.items(), key=lambda x: x[1])
    print(f"\n Best model on new data: {best_model[0]} ({best_model[1]*100:.2f}%)")

    print(f"\n Performance Comparison:")
    print(f"Original LightGBM accuracy: 95.41%")
    print(f"New data LightGBM accuracy: {accuracy_results['LightGBM']*100:.2f}%")

    change = accuracy_results['LightGBM'] - 0.9541
    if change > 0:
        print(f"Improvement: +{change*100:.2f}%")
    else:
        print(f" Drop: {change*100:.2f}%")

else:
    print(" No labels available for accuracy calculation")
    print(" Prediction distribution across all models:")
    for model_name in models.keys():
        pred_counts = new_df[f'pred_{model_name}'].value_counts(normalize=True)
        print(f"{model_name:<20}: Fake {pred_counts.get(0, 0):.1%} | Real {pred_counts.get(1, 0):.1%}")

Evaluating results...

 Accuracy on New Data:
Naive_Bayes         : 0.7334 (73.34%)
Random_Forest       : 0.7614 (76.14%)
LightGBM            : 0.7555 (75.55%)
Complement_NB       : 0.7045 (70.45%)
Passive_Aggressive  : 0.7451 (74.51%)

 Best model on new data: Random_Forest (76.14%)

 Performance Comparison:
Original LightGBM accuracy: 95.41%
New data LightGBM accuracy: 75.55%
 Drop: -19.86%


In [7]:
print("Saving results...")

output_file = 'final_bn_data_predictions.csv'
new_df.to_csv(output_file, index=False, encoding='utf-8')
print(f" Predictions saved to: {output_file}")

# Save summary
summary = {
    'dataset_size': len(new_df),
    'models_tested': list(models.keys()),
    'has_labels': 'label' in new_df.columns
}

if 'label' in new_df.columns:
    summary['accuracies'] = {k: float(v) for k, v in accuracy_results.items()}  # Convert to float for JSON
    summary['best_model'] = best_model[0]

import json
with open('final_bn_data_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Summary saved to: final_bn_data_summary.json")

print(f"\n Testing complete! Check '{output_file}' for all predictions.")

Saving results...
 Predictions saved to: final_bn_data_predictions.csv
Summary saved to: final_bn_data_summary.json

 Testing complete! Check 'final_bn_data_predictions.csv' for all predictions.


In [8]:
# Start Retraining
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import time

print("Libraries imported for retraining")

Libraries imported for retraining


In [9]:
# Creating Combined Dataset
print(" CREATING COMBINED DATASET...")

original_df = pd.read_csv('../data/processed_data.csv')
print(f"Original data: {original_df.shape}")
print(f"Original label distribution:\n{original_df['label'].value_counts()}")

# Preparing new dataset
new_df_renamed = new_df.rename(columns={'content': 'text'})

# Combining
combined_df = pd.concat([
    original_df[['text', 'label']],
    new_df_renamed[['text', 'label']]
], ignore_index=True)

print(f"New data: {new_df_renamed.shape}")
print(f"Combined data: {combined_df.shape}")
print(f"Combined label distribution:\n{combined_df['label'].value_counts()}")

print(f"Label types before cleaning: {combined_df['label'].unique()}")
combined_df['label'] = combined_df['label'].astype(int)
print(f"Label types after cleaning: {combined_df['label'].unique()}")

 CREATING COMBINED DATASET...
Original data: (8501, 8)
Original label distribution:
label
1    7202
0    1299
Name: count, dtype: int64
New data: (14537, 9)
Combined data: (23038, 2)
Combined label distribution:
label
1.0    17202
0.0     5836
Name: count, dtype: int64
Label types before cleaning: [0. 1.]
Label types after cleaning: [0 1]


In [10]:
print(" CREATING NEW VECTORIZER ON COMBINED DATA...")

from sklearn.feature_extraction.text import TfidfVectorizer

new_vectorizer = TfidfVectorizer(
    max_features=6000,
    lowercase=False,
    min_df=3,
    max_df=0.85,
    ngram_range=(1, 2)
)

X_combined = new_vectorizer.fit_transform(combined_df['text'])
y_combined = combined_df['label']

print(f"Combined vectorized shape: {X_combined.shape}")
print(f"Vocabulary size: {len(new_vectorizer.get_feature_names_out())}")

joblib.dump(new_vectorizer, 'combined_vectorizer.pkl')

 CREATING NEW VECTORIZER ON COMBINED DATA...
Combined vectorized shape: (23038, 6000)
Vocabulary size: 6000


['combined_vectorizer.pkl']

In [11]:
print("RETRAINING MODELS ON COMBINED DATA...")
from sklearn.model_selection import train_test_split

X_train_combined, X_test_combined, y_train_combined, y_test_combined = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

print(f"Training set: {X_train_combined.shape}")
print(f"Test set: {X_test_combined.shape}")
print(f"Training labels: {pd.Series(y_train_combined).value_counts().to_dict()}")
print(f"Test labels: {pd.Series(y_test_combined).value_counts().to_dict()}")

import time
retrained_models = {}
training_times = {}

print("\n1. Retraining Naive Bayes...")
start_time = time.time()
nb_combined = MultinomialNB()
nb_combined.fit(X_train_combined, y_train_combined)
training_times['Naive_Bayes'] = time.time() - start_time
retrained_models['Naive_Bayes_Combined'] = nb_combined

print("2. Retraining Random Forest...")
start_time = time.time()
rf_combined = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    max_depth=15
)
rf_combined.fit(X_train_combined, y_train_combined)
training_times['Random_Forest'] = time.time() - start_time
retrained_models['Random_Forest_Combined'] = rf_combined

print("3. Retraining LightGBM...")
start_time = time.time()
lgb_combined = LGBMClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    verbose=0,
    max_depth=10
)
lgb_combined.fit(X_train_combined, y_train_combined)
training_times['LightGBM'] = time.time() - start_time
retrained_models['LightGBM_Combined'] = lgb_combined

print("4. Retraining Complement NB...")
start_time = time.time()
cnb_combined = ComplementNB()
cnb_combined.fit(X_train_combined, y_train_combined)
training_times['Complement_NB'] = time.time() - start_time
retrained_models['Complement_NB_Combined'] = cnb_combined

print("5. Retraining Passive Aggressive...")
start_time = time.time()
pa_combined = PassiveAggressiveClassifier(
    max_iter=1500,
    random_state=42,
    verbose=0
)
pa_combined.fit(X_train_combined, y_train_combined)
training_times['Passive_Aggressive'] = time.time() - start_time
retrained_models['Passive_Aggressive_Combined'] = pa_combined

print("All models retrained on combined data")
print(f"Training times: {training_times}")

RETRAINING MODELS ON COMBINED DATA...
Training set: (18430, 6000)
Test set: (4608, 6000)
Training labels: {1: 13761, 0: 4669}
Test labels: {1: 3441, 0: 1167}

1. Retraining Naive Bayes...
2. Retraining Random Forest...
3. Retraining LightGBM...
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Ligh

In [12]:
print(" EVALUATING RETRAINED MODELS ON COMBINED TEST SET...")

combined_test_results = {}
print("Performance on Combined Test Set (20% holdout):")
print("=" * 60)

for model_name, model in retrained_models.items():
    y_pred = model.predict(X_test_combined)
    accuracy = accuracy_score(y_test_combined, y_pred)

    cm = confusion_matrix(y_test_combined, y_pred)
    precision = cm[1,1] / (cm[1,1] + cm[0,1]) if (cm[1,1] + cm[0,1]) > 0 else 0
    recall = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    combined_test_results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1
    }

    print(f"{model_name:<25} | Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

# Best model
best_combined_model = max(combined_test_results.items(), key=lambda x: x[1]['accuracy'])
print(f"\n BEST MODEL ON COMBINED TEST: {best_combined_model[0]}")
print(f"   Accuracy: {best_combined_model[1]['accuracy']:.4f} ({best_combined_model[1]['accuracy']*100:.2f}%)")

 EVALUATING RETRAINED MODELS ON COMBINED TEST SET...
Performance on Combined Test Set (20% holdout):
Naive_Bayes_Combined      | Accuracy: 0.7973 | Precision: 0.8213 | Recall: 0.9311 | F1: 0.8728
Random_Forest_Combined    | Accuracy: 0.7928 | Precision: 0.7833 | Recall: 0.9988 | F1: 0.8780
LightGBM_Combined         | Accuracy: 0.8188 | Precision: 0.8206 | Recall: 0.9692 | F1: 0.8887
Complement_NB_Combined    | Accuracy: 0.7077 | Precision: 0.8339 | Recall: 0.7600 | F1: 0.7952
Passive_Aggressive_Combined | Accuracy: 0.6864 | Precision: 0.8103 | Recall: 0.7573 | F1: 0.7829

 BEST MODEL ON COMBINED TEST: LightGBM_Combined
   Accuracy: 0.8188 (81.88%)


C:\Users\imtia\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
print(" TESTING RETRAINED MODELS ON NEW DATASET...")


X_new_combined = new_vectorizer.transform(new_df_renamed['text'])
print(f"New dataset transformed: {X_new_combined.shape}")

new_dataset_results = {}
print("\nPerformance on Original New Dataset:")
print("=" * 50)

for model_name, model in retrained_models.items():
    y_pred_new = model.predict(X_new_combined)
    accuracy = accuracy_score(new_df_renamed['label'], y_pred_new)

    new_dataset_results[model_name] = accuracy
    print(f"{model_name:<25}: {accuracy:.4f} ({accuracy*100:.2f}%)")

print(f"\n PERFORMANCE COMPARISON:")
original_best = 0.7614
retrained_best = max(new_dataset_results.values())
improvement = retrained_best - original_best

print(f"Original models (best): {original_best*100:.2f}%")
print(f"Retrained models (best): {retrained_best*100:.2f}%")
print(f"Improvement: {improvement*100:+.2f}%")

 TESTING RETRAINED MODELS ON NEW DATASET...
New dataset transformed: (14537, 6000)

Performance on Original New Dataset:
Naive_Bayes_Combined     : 0.7336 (73.36%)
Random_Forest_Combined   : 0.7340 (73.40%)
LightGBM_Combined        : 0.7806 (78.06%)
Complement_NB_Combined   : 0.6851 (68.51%)
Passive_Aggressive_Combined: 0.8390 (83.90%)

 PERFORMANCE COMPARISON:
Original models (best): 76.14%
Retrained models (best): 83.90%
Improvement: +7.76%


C:\Users\imtia\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
print(" SAVING RETRAINED MODELS AND RESULTS...")

for model_name, model in retrained_models.items():
    clean_name = model_name.replace('_Combined', '').lower()
    filename = f"retrained_{clean_name}.pkl"
    joblib.dump(model, filename)
    print(f"Saved: {filename}")

results_summary = {
    'dataset_info': {
        'original_size': original_df.shape[0],
        'new_size': new_df.shape[0],
        'combined_size': combined_df.shape[0],
        'training_size': X_train_combined.shape[0],
        'test_size': X_test_combined.shape[0]
    },
    'combined_test_performance': combined_test_results,
    'new_dataset_performance': new_dataset_results,
    'training_times': training_times,
    'improvement_analysis': {
        'original_best_accuracy': original_best,
        'retrained_best_accuracy': retrained_best,
        'improvement': improvement,
        'improvement_percentage': improvement * 100
    }
}

with open('retraining_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(" Results saved to 'retraining_results.json'")

print(f"\n RETRAINING COMPLETE!")
print(f" New files created:")
print(f"   - combined_vectorizer.pkl")
print(f"   - retrained_*.pkl (5 model files)")
print(f"   - retraining_results.json")

 SAVING RETRAINED MODELS AND RESULTS...
Saved: retrained_naive_bayes.pkl
Saved: retrained_random_forest.pkl
Saved: retrained_lightgbm.pkl
Saved: retrained_complement_nb.pkl
Saved: retrained_passive_aggressive.pkl
 Results saved to 'retraining_results.json'

 RETRAINING COMPLETE!
 New files created:
   - combined_vectorizer.pkl
   - retrained_*.pkl (5 model files)
   - retraining_results.json
